In [2]:
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

In [3]:
df = pd.read_csv(
    r"D:\下载2\Books_rating\Books_rating.csv",
    nrows=1000
)

# 清理数据
df = df.dropna(subset=["Title", "review/summary", "review/text", "review/score", "User_id", "Id"])
df = df.reset_index(drop=True)
# 合并文本：标题 + 摘要 + 评论（内容推荐核心）
df["content"] = df["Title"] + " " + df["review/summary"] + " " + df["review/text"]

# 评分归一化（0~5分 → 0~1权重）
df["rating_weight"] = df["review/score"] / 5.0

# ===================== 2. TF-IDF 向量化 =====================

tfidf = TfidfVectorizer(stop_words="english", max_features=5000)
tfidf_matrix = tfidf.fit_transform(df["content"])

In [4]:
# ===================== 3. 内容相似度（带评分加权） =====================
def content_based_recommend(book_id, top_n=5):
    """
    内容推荐：根据书籍内容 + 评分权重做相似推荐
    """
    # 找到这本书的索引
    idx = df[df["Id"] == book_id].index[0]

    # 计算余弦相似度
    sim_scores = cosine_similarity(tfidf_matrix[idx], tfidf_matrix).flatten()

    # 评分加权：高评分书权重更高
    sim_scores = sim_scores * df["rating_weight"].values

    # 获取TopN（排除自己）
    sim_indices = np.argsort(sim_scores)[::-1][1:top_n+1]
    sim_scores = sim_scores[sim_indices]

    # 构造结果
    recs = df.iloc[sim_indices][["Id", "Title", "review/score"]].copy()
    recs["content_score"] = np.round(sim_scores, 4)

    return recs

# 测试：给第一本书做内容推荐
if __name__ == "__main__":
    test_book_id = df["Id"].iloc[0]
    print("测试书籍：", df[df["Id"] == test_book_id]["Title"].values[0])
    content_recs = content_based_recommend(test_book_id, top_n=5)
    print("\n===== 内容过滤推荐结果 =====")
    print(content_recs)

测试书籍： Its Only Art If Its Well Hung!

===== 内容过滤推荐结果 =====
             Id                                              Title  \
121  0789480662                  Eyewitness Travel Guide to Europe   
690  B000MCKQRS  Cruel and Unusual (G K Hall Large Print Book S...   
733  087474721X  Engendering Culture: Manhood and Womanhood In ...   
474  B0000630MU                       HTML: The Complete Reference   
477  B0000630MU                       HTML: The Complete Reference   

     review/score  content_score  
121           5.0         0.0857  
690           4.0         0.0851  
733           5.0         0.0782  
474           5.0         0.0767  
477           5.0         0.0747  


# 协同过滤推荐

In [5]:
import pandas as pd
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity

# ===================== 1. 读取数据 =====================
df = pd.read_csv(
    r"D:\下载2\Books_rating\Books_rating.csv",
    nrows=1000
)
df = df.dropna(subset=["User_id", "Id", "review/score"])
df = df.reset_index(drop=True)

# 构建用户-物品评分矩阵
user_item_matrix = df.pivot_table(
    index="User_id",
    columns="Id",
    values="review/score"
).fillna(0)

# ===================== 2. 用户协同过滤 =====================
def user_based_recommend(user_id, top_n=5):
    if user_id not in user_item_matrix.index:
        return pd.DataFrame()

    # 计算用户相似度
    user_sim = cosine_similarity(user_item_matrix)
    user_idx = list(user_item_matrix.index).index(user_id)
    sim_users = np.argsort(user_sim[user_idx])[::-1][1:10]  # 取最相似10个用户

    # 加权评分
    rec_scores = user_item_matrix.iloc[sim_users].mean(axis=0)
    rec_scores = rec_scores.sort_values(ascending=False)

    # 排除已评分书籍
    user_rated = user_item_matrix.iloc[user_idx]
    rec_scores = rec_scores[user_rated == 0]

    # 构造结果
    recs = pd.DataFrame({
        "Id": rec_scores.index[:top_n],
        "collab_score": np.round(rec_scores.values[:top_n], 4)
    })
    return recs

# ===================== 3. 物品协同过滤 =====================
def item_based_recommend(book_id, top_n=5):
    if book_id not in user_item_matrix.columns:
        return pd.DataFrame()

    # 计算物品相似度
    item_sim = cosine_similarity(user_item_matrix.T)
    item_idx = list(user_item_matrix.columns).index(book_id)
    sim_scores = item_sim[item_idx]

    # 排序
    sim_indices = np.argsort(sim_scores)[::-1][1:top_n+1]
    sim_scores = sim_scores[sim_indices]

    recs = pd.DataFrame({
        "Id": [user_item_matrix.columns[i] for i in sim_indices],
        "collab_score": np.round(sim_scores, 4)
    })
    return recs

# 测试
if __name__ == "__main__":
    test_user = df["User_id"].iloc[0]
    test_book = df["Id"].iloc[0]

    print("===== 用户协同推荐 =====")
    print(user_based_recommend(test_user, 5))

    print("\n===== 物品协同推荐 =====")
    print(item_based_recommend(test_book, 5))

===== 用户协同推荐 =====
           Id  collab_score
0  B0000630MU        1.1111
1  B000MCKQRS        1.0000
2  B0006D6DRK        0.5556
3  1861081162        0.5556
4  B0000CJHIO        0.5556

===== 物品协同推荐 =====
           Id  collab_score
0  B000PHTCGG           0.0
1  0971728429           0.0
2  0792391810           0.0
3  0802422772           0.0
4  0802841899           0.0


# 混合推荐算法

In [7]:
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# ===================== 1. 全局加载数据（只读一次，避免报错） =====================
df = pd.read_csv(
    r"D:\下载2\Books_rating\Books_rating.csv",
    nrows=1000  # 只读取前1000行，速度快
)

# 清理空值
df = df.dropna(subset=["Title", "review/summary", "review/text", "review/score", "User_id", "Id"])
df = df.reset_index(drop=True)

# 合并文本（标题+摘要+评论）
df["content"] = df["Title"] + " " + df["review/summary"] + " " + df["review/text"]

# 评分权重（0~1）
df["rating_weight"] = df["review/score"] / 5.0

# TF-IDF 向量化
tfidf = TfidfVectorizer(stop_words="english", max_features=5000)
tfidf_matrix = tfidf.fit_transform(df["content"])

# 构建用户-物品评分矩阵
user_item_matrix = df.pivot_table(
    index="User_id",
    columns="Id",
    values="review/score"
).fillna(0)

# ===================== 2. 内容过滤推荐 =====================
def content_based_recommend(book_id, top_n=5):
    try:
        idx = df[df["Id"] == book_id].index[0]
    except:
        return pd.DataFrame()

    # 计算相似度
    sim_scores = cosine_similarity(tfidf_matrix[idx], tfidf_matrix).flatten()
    sim_scores = sim_scores * df["rating_weight"].values  # 评分加权

    # 排序
    sim_indices = np.argsort(sim_scores)[::-1][1:top_n+1]
    sim_scores = sim_scores[sim_indices]

    recs = df.iloc[sim_indices][["Id", "Title", "review/score"]].copy()
    recs["content_score"] = np.round(sim_scores, 4)
    return recs

# ===================== 3. 物品协同过滤推荐 =====================
def item_based_recommend(book_id, top_n=5):
    if book_id not in user_item_matrix.columns:
        return pd.DataFrame()

    item_sim = cosine_similarity(user_item_matrix.T)
    item_idx = list(user_item_matrix.columns).index(book_id)
    sim_scores = item_sim[item_idx]

    sim_indices = np.argsort(sim_scores)[::-1][1:top_n+1]
    sim_scores = sim_scores[sim_indices]

    recs = pd.DataFrame({
        "Id": [user_item_matrix.columns[i] for i in sim_indices],
        "collab_score": np.round(sim_scores, 4)
    })
    return recs

# ===================== 4. 混合推荐（核心！） =====================
def hybrid_recommend(
    book_id,
    content_weight=0.5,
    collab_weight=0.5,
    top_n=5
):
    # 内容推荐
    content_recs = content_based_recommend(book_id, top_n=10)

    # 协同推荐
    collab_recs = item_based_recommend(book_id, top_n=10)

    # 合并
    merged = pd.merge(content_recs, collab_recs, on="Id", how="outer").fillna(0)

    # 最终得分
    merged["final_score"] = (
        merged["content_score"] * content_weight +
        merged["collab_score"] * collab_weight
    )

    # 排序输出
    merged = merged.sort_values("final_score", ascending=False).head(top_n)
    merged = merged.reset_index(drop=True)
    return merged[["Id", "Title", "review/score", "final_score"]]

# ===================== 测试运行 =====================
if __name__ == "__main__":
    # 取第一本书做测试
    test_book_id = df["Id"].iloc[0]
    test_book_title = df[df["Id"] == test_book_id]["Title"].values[0]

    print("当前推荐书籍：", test_book_title)
    print("\n===== 混合推荐结果（内容50% + 协同50%） =====")

    # 运行推荐
    result = hybrid_recommend(test_book_id, content_weight=0.5, collab_weight=0.5, top_n=5)
    print(result)

当前推荐书籍： Its Only Art If Its Well Hung!

===== 混合推荐结果（内容50% + 协同50%） =====
           Id                                              Title  \
0  0789480662                  Eyewitness Travel Guide to Europe   
1  B000MCKQRS  Cruel and Unusual (G K Hall Large Print Book S...   
2  087474721X  Engendering Culture: Manhood and Womanhood In ...   
3  B0000630MU                       HTML: The Complete Reference   
4  B0000630MU                       HTML: The Complete Reference   

   review/score  final_score  
0           5.0      0.04285  
1           4.0      0.04255  
2           5.0      0.03910  
3           5.0      0.03835  
4           5.0      0.03735  
